# LECTURE 2: DATA TRANSFORMATION
### *Scaling, Categorical Encoding, Distribution Transforms, Feature Expansion, and Target Transformation*
**Course:** Big Data Analytics in Finance (Phân tích Dữ liệu Lớn trong Tài chính)  
**Institution:** School of Applied Mathematics and Economics (SAMI / MFE) - National Economics University (NEU)  
**Instructor:** Dr. Nguyen Tuan Long  
**Reference Slides:** `Slides/04_L02_Data_transform.pdf`

---

## Executive Summary & Learning Objectives
Most machine learning algorithms cannot learn effectively from raw, unprocessed data:
1. **Scale Sensitivity:** Distance-based models (KNN, SVM, K-Means) and gradient-based algorithms (Neural Networks, Logistic Regression) are heavily distorted when features have differing scales (e.g., Annual Income $\$50,000$ vs. Experience $5$ years).
2. **Text Labels:** Algorithms require numeric matrices, necessitating principled categorical encodings (Nominal vs. Ordinal).
3. **Non-Gaussianity & Heavy Tails:** Financial series (trading volume, order sizes, firm asset valuations) exhibit extreme positive skewness and fat tails that violate linear model assumptions.
4. **Target Skewness:** Severe heteroskedasticity and outlier dominance can be resolved by modeling $f(y)$ rather than $y$ directly.

**Core Principle (Zero Data Leakage):**
> *Never fit a transformer on test or validation data! Always fit transformers on training data only (`scaler.fit(X_train)`), and apply the fitted transformer (`scaler.transform()`) to validation, test, and production data.*

---


# PART I: FEATURE TRANSFORMATION
---

## 1. Taxonomy of Data Variables
Before choosing a transformation strategy, we must diagnose the statistical type of each feature:

```
                      +-------------------+
                      |   DATA VARIABLE   |
                      +---------+---------+
                                |
                +---------------+---------------+
                |                               |
        +-------v-------+               +-------v-------+
        |   NUMERICAL   |               |  CATEGORICAL  |
        +-------+-------+               +-------+-------+
                |                               |
        +-------+-------+               +-------+-------+-------+
        |               |               |               |       |
+-------v-------+ +-----v-----+  +------v------+ +------v------+ +v--------+
|    INTEGER    | |   FLOAT   |  |   NOMINAL   | |   ORDINAL   | | BOOLEAN |
| (Order Count) | | (Returns) |  | (Exchange)  | |(Credit Rate)| |(Default)|
+---------------+ +-----------+  +-------------+ +-------------+ +---------+
```

---

## 2. Scaling Numerical Data: Bringing Features to a Comparable Scale
When features have disparate physical units or variances, unscaled distances are dominated by features with large absolute magnitudes.

### 2.1 Technique 1: MinMaxScaler
**Mathematical Formulation:**
$$x_{\text{scaled}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \in [0, 1]$$
- **Properties:** Preserves exact original distribution shape; shifts and scales values to bounded $[0, 1]$ interval.
- **Vulnerability:** Highly sensitive to outliers. A single extreme whale order compresses $99\%$ of regular trades into a narrow band near $0$.
- **New Range Test:** If test data contains values outside $[x_{\min}, x_{\max}]$, $x_{\text{scaled}}$ will fall outside $[0, 1]$ unless clipped (`clip=True`).

#### Hand Calculation Drill:
Given training values $x = (20, 30, 50)$ (income in USD thousands):
- $x_{\min} = 20, \quad x_{\max} = 50, \quad \text{Range} = 30$
- $x = 20 \implies x' = \frac{20 - 20}{30} = 0.00$
- $x = 30 \implies x' = \frac{30 - 20}{30} = \frac{10}{30} \approx 0.333$
- $x = 50 \implies x' = \frac{50 - 20}{30} = \frac{30}{30} = 1.00$
- *Unseen Test Value:* $x_{\text{test}} = 60 \implies x_{\text{test}}' = \frac{60 - 20}{30} = 1.333$ (exceeds $1.0$).


In [ ]:
# 1. MinMaxScaler Hands-on Implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

# Deterministic random seed
np.random.seed(42)

# Training and test observations
train_data = np.array([20, 30, 50]).reshape(-1, 1)
test_data = np.array([60]).reshape(-1, 1)

# Fit strictly on train
minmax = MinMaxScaler()
train_minmax = minmax.fit_transform(train_data)
test_minmax = minmax.transform(test_data)

print("=== MinMaxScaler Verification ===")
print("Learned Data Min :", minmax.data_min_[0])
print("Learned Data Max :", minmax.data_max_[0])
print("Train Scaled     :", train_minmax.ravel())
print("Test Scaled (60) :", test_minmax.ravel(), "-> Notice value > 1.0!")


### 2.2 Technique 2: StandardScaler (Z-Score Normalization)
**Mathematical Formulation:**
$$z = \frac{x - \mu}{\sigma}$$
where $\mu = \frac{1}{N}\sum_{i=1}^N x_i$ and $\sigma = \sqrt{\frac{1}{N}\sum_{i=1}^N (x_i - \mu)^2}$.
- **Properties:** Centers data at mean zero ($\mu=0$) and scales to unit variance ($\sigma=1$).
- **Crucial Didactic Warning:** *Mean zero does NOT mean Gaussian!* If the raw data is heavily skewed or multimodal, the standardized variable remains identically skewed and multimodal. It only translates and rescales the axes.

#### Hand Calculation Drill:
Given training values $x = (2, 4, 6)$:
- Mean $\mu = \frac{2 + 4 + 6}{3} = 4.0$
- Variance $\sigma^2 = \frac{(2-4)^2 + (4-4)^2 + (6-4)^2}{3} = \frac{4 + 0 + 4}{3} = \frac{8}{3} \approx 2.667$
- Standard Deviation $\sigma = \sqrt{2.667} \approx 1.633$
- $x = 2 \implies z = \frac{2 - 4}{1.633} = -1.225$
- $x = 4 \implies z = \frac{4 - 4}{1.633} = 0.000$
- $x = 6 \implies z = \frac{6 - 4}{1.633} = +1.225$
- *Unseen Test Value:* $x_{\text{test}} = 8 \implies z_{\text{test}} = \frac{8 - 4}{1.633} \approx +2.450$


In [ ]:
# 2. StandardScaler Hands-on Implementation
train_std_data = np.array([2, 4, 6]).reshape(-1, 1)
test_std_data = np.array([8]).reshape(-1, 1)

std_scaler = StandardScaler()
train_std = std_scaler.fit_transform(train_std_data)
test_std = std_scaler.transform(test_std_data)

print("=== StandardScaler Verification ===")
print("Learned Mean     :", std_scaler.mean_[0])
print("Learned Std Dev  :", std_scaler.scale_[0])
print("Train Standardized:", train_std.ravel())
print("Test Standardized :", test_std.ravel())


### 2.3 Technique 3: RobustScaler (Median and IQR)
When data contains extreme financial anomalies (fat tails, flash crashes, sovereign debt spikes), the sample mean and standard deviation are severely distorted.

**Mathematical Formulation:**
$$x_{\text{robust}} = \frac{x - Q_2(x)}{Q_3(x) - Q_1(x)} = \frac{x - \text{median}(x)}{\text{IQR}(x)}$$
- **Properties:** Centers around the median and scales by the Interquartile Range ($25\text{th}$ to $75\text{th}$ percentile).
- **Core Distinction:** RobustScaler is a *robust scale*, NOT an *outlier removal* algorithm. Outliers still remain extreme, but they cannot distort the scaling of the central $50\%$ of observations.

#### Hand Calculation Drill:
Given observations containing an outlier: $x = (10, 12, 14, 16, 100)$:
- Median ($Q_2$) = $14$
- First Quartile ($Q_1$) = $12$, Third Quartile ($Q_3$) = $16$
- $\text{IQR} = 16 - 12 = 4$
- $x = 10 \implies x' = \frac{10 - 14}{4} = -1.00$
- $x = 14 \implies x' = \frac{14 - 14}{4} = 0.00$
- $x = 16 \implies x' = \frac{16 - 14}{4} = +0.50$
- $x = 100 \implies x' = \frac{100 - 14}{4} = +21.50$ (Outlier clearly retained at $+21.5\times\text{IQR}$!)


In [ ]:
# 3. RobustScaler Hands-on Implementation
train_outlier_data = np.array([10, 12, 14, 16, 100]).reshape(-1, 1)

rob_scaler = RobustScaler()
train_rob = rob_scaler.fit_transform(train_outlier_data)

print("=== RobustScaler Verification ===")
print("Learned Center (Median):", rob_scaler.center_[0])
print("Learned Scale (IQR)    :", rob_scaler.scale_[0])
print("Scaled Data (Outlier preserved at 21.5):", train_rob.ravel())


### 2.4 Scaler Decision Matrix
| Scaler | Primary Use Cases | Key Limitation / Hazard |
| :--- | :--- | :--- |
| **MinMaxScaler** | Bounded $[0, 1]$ models (Neural Nets, Image Tensors, KNN). | Extreme sensitivity to outliers; new test points may exceed boundaries. |
| **StandardScaler** | Linear Regression, Logistic Regression, SVM, PCA. | Outliers inflate variance; does NOT convert skewed data into normal distribution. |
| **RobustScaler** | Heavy-tailed financial distributions (P/E ratios, trading volumes). | Unbounded output; not suitable for models requiring bounded inputs. |

---

## 3. Full-Data Practice 1: Diabetes Dataset Benchmark
We evaluate the impact of different scalers on a downstream regression model predicting disease progression.


In [ ]:
# Full-Data Practice 1: Diabetes Dataset Scaler Benchmark
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score

# Load clinical dataset (442 samples, 10 physiological features)
X_diab, y_diab = load_diabetes(return_X_y=True)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_diab, y_diab, test_size=0.25, random_state=42
)

# Benchmark dictionary
scalers = {
    "No Scaling (Raw)": None,
    "MinMaxScaler": MinMaxScaler(),
    "StandardScaler": StandardScaler(),
    "RobustScaler": RobustScaler()
}

print(f"{'Scaler Applied':<20} | {'KNN R2 Score':<14} | {'Ridge R2 Score':<14}")
print("-" * 54)

for name, scaler in scalers.items():
    if scaler is None:
        X_tr_s, X_te_s = X_train_d, X_test_d
    else:
        X_tr_s = scaler.fit_transform(X_train_d)
        X_te_s = scaler.transform(X_test_d)
    
    # KNN (Distance-based: very sensitive to scaling)
    knn = KNeighborsRegressor(n_neighbors=7).fit(X_tr_s, y_train_d)
    knn_r2 = knn.score(X_te_s, y_test_d)
    
    # Ridge (L2 regularized: scale affects shrinkage penalty)
    ridge = Ridge(alpha=1.0).fit(X_tr_s, y_train_d)
    ridge_r2 = ridge.score(X_te_s, y_test_d)
    
    print(f"{name:<20} | {knn_r2:<14.4f} | {ridge_r2:<14.4f}")


---

## 4. Encoding Categorical Data: Ordinal vs. Nominal Encodings
Machine learning estimators cannot compute arithmetic loss functions on strings or text symbols.

### 4.1 Categorical Foundations
1. **Ordinal Categories:** Categories possess a natural, indisputable hierarchy ($A < B < C$ or $\text{Low} < \text{Medium} < \text{High}$).
   - **Method:** `OrdinalEncoder` assigns ordered integers ($0, 1, 2, \dots$).
2. **Nominal Categories:** Categories have NO intrinsic ranking (Country: US, UK, DE; Stock Sector: Tech, Energy, Finance).
   - **Method:** `OneHotEncoder` creates $K$ binary indicator columns.
   - **Dummy Variable Trap:** In unregularized linear models, $K$ mutually exclusive dummies sum to $1$, causing exact collinearity with the intercept ($X_1 + X_2 + \dots + X_K = 1$). Set `drop='first'` to create $K-1$ indicators.

#### Hand Calculation Drill: Toy Customer Dataset
| Customer | Credit Risk (Ordinal) | Region (Nominal) |
| :--- | :--- | :--- |
| **C1** | Low | North |
| **C2** | Medium | South |
| **C3** | High | North |
| **C4** | Medium | Central |

- **Ordinal Risk Encoding:** $\text{Low} \mapsto 0, \; \text{Medium} \mapsto 1, \; \text{High} \mapsto 2 \implies (0, 1, 2, 1)^T$.
- **One-Hot Region Encoding:** Columns = `[Central, North, South]`
  - C1 (North)   $\mapsto [0, 1, 0]$
  - C2 (South)   $\mapsto [0, 0, 1]$
  - C3 (North)   $\mapsto [0, 1, 0]$
  - C4 (Central) $\mapsto [1, 0, 0]$


In [ ]:
# 4. OrdinalEncoder & OneHotEncoder Implementation
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

df_customers = pd.DataFrame({
    'CustomerID': ['C1', 'C2', 'C3', 'C4'],
    'CreditRisk': ['Low', 'Medium', 'High', 'Medium'],
    'Region': ['North', 'South', 'North', 'Central']
})

# 1. Ordinal Encoding with explicit hierarchy
risk_hierarchy = [['Low', 'Medium', 'High']]
ord_enc = OrdinalEncoder(categories=risk_hierarchy)
df_customers['Risk_Encoded'] = ord_enc.fit_transform(df_customers[['CreditRisk']])

# 2. One-Hot Encoding with drop='first' (prevents dummy variable trap)
ohe = OneHotEncoder(sparse_output=False, drop='first')
region_ohe = ohe.fit_transform(df_customers[['Region']])
ohe_cols = ohe.get_feature_names_out(['Region'])
df_ohe = pd.DataFrame(region_ohe, columns=ohe_cols)

print("=== Categorical Encoding Result ===")
print(pd.concat([df_customers, df_ohe], axis=1))


### 4.2 Full-Data Practice 2: Categorical Encoding on Clinical Breast Cancer Data
We construct a pipeline handling mixed categorical variables with missing categories using `ColumnTransformer`.


In [ ]:
# Full-Data Practice 2: Clinical Categorical Data via ColumnTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Realistic diagnostic DataFrame with ordinal, nominal, and missing entries
raw_cancer_df = pd.DataFrame({
    'age_group': ['40-49', '50-59', '60-69', '40-49', '50-59', '30-39', '60-69', '50-59'],
    'tumor_size': ['15-19', '25-29', '35-39', '20-24', np.nan, '10-14', '30-34', '20-24'],
    'deg_malig': ['1', '2', '3', '2', '3', '1', '2', '3'], # Ordinal malignancy
    'menopause': ['premeno', 'ge40', 'lt40', 'ge40', 'premeno', 'premeno', 'ge40', 'ge40'],
    'recurrence': [0, 1, 1, 0, 1, 0, 0, 1]
})

X_cat = raw_cancer_df.drop('recurrence', axis=1)
y_cat = raw_cancer_df['recurrence']

# Pipelines for ordinal vs nominal branches
ord_features = ['deg_malig']
nom_features = ['age_group', 'tumor_size', 'menopause']

nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('ord', OrdinalEncoder(), ord_features),
    ('nom', nom_pipeline, nom_features)
])

clf_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(random_state=42))
])

clf_pipe.fit(X_cat, y_cat)
print("Pipeline Successfully Trained!")
print("Transformed Feature Count:", clf_pipe.named_steps['prep'].transform(X_cat).shape[1])


# PART II: DISTRIBUTION & ADVANCED TRANSFORMS
---

## 5. Distribution Transforms: Reshaping Feature Geometry
Many linear models, Gaussian Naive Bayes, and statistical significance tests assume features follow approximately normal distributions with homogeneous variance.

### 5.1 PowerTransformer (Box-Cox and Yeo-Johnson)
Power transformations stabilize variance and convert skewed distributions into bell-shaped Gaussian representations.

**Box-Cox Transformation (Strictly Positive $x > 0$):**
$$x^{(\lambda)} = \begin{cases} \frac{x^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0 \\ \ln(x) & \text{if } \lambda = 0 \end{cases}$$

**Yeo-Johnson Transformation (Handles Positive, Zero, and Negative Values):**
$$x^{(\lambda)} = \begin{cases}
\frac{(x+1)^\lambda - 1}{\lambda} & \text{if } \lambda \neq 0, x \ge 0 \\
\ln(x+1) & \text{if } \lambda = 0, x \ge 0 \\
-\frac{(-x+1)^{2-\lambda} - 1}{2-\lambda} & \text{if } \lambda \neq 2, x < 0 \\
-\ln(-x+1) & \text{if } \lambda = 2, x < 0
\end{cases}$$
The optimal parameter $\hat{\lambda}$ is estimated via Maximum Likelihood Estimation (MLE).

#### Hand Calculation Drill:
Given right-skewed insurance claims $y = (1, 4, 9, 25, 100)$ with $\lambda = 0.5$:
$$y^{(0.5)} = \frac{y^{0.5} - 1}{0.5} = 2(\sqrt{y} - 1)$$
- $y = 1 \implies 2(\sqrt{1} - 1) = 0$
- $y = 4 \implies 2(\sqrt{4} - 1) = 2(2 - 1) = 2$
- $y = 9 \implies 2(\sqrt{9} - 1) = 2(3 - 1) = 4$
- $y = 25 \implies 2(\sqrt{25} - 1) = 2(5 - 1) = 8$
- $y = 100 \implies 2(\sqrt{100} - 1) = 2(10 - 1) = 18$
- *Original Spread:* $100 - 25 = 75$.  
- *Transformed Spread:* $18 - 8 = 10$. The fat right tail is compressed dramatically!


In [ ]:
# 5. PowerTransformer Verification & Visualization
from sklearn.preprocessing import PowerTransformer

claims = np.array([1, 4, 9, 25, 100]).reshape(-1, 1)

# Fit Yeo-Johnson (standardize=False to inspect pure power transform)
pt = PowerTransformer(method='yeo-johnson', standardize=False)
claims_pt = pt.fit_transform(claims)

print("=== PowerTransformer Results ===")
print("Estimated Optimal Lambda (lambda):", np.round(pt.lambdas_[0], 4))
print("Raw Claims                 :", claims.ravel())
print("Power-Transformed Claims   :", np.round(claims_pt.ravel(), 4))


### 5.2 QuantileTransformer (Rank-Based Normalization)
Non-linear transform mapping the empirical cumulative distribution function (ECDF) to a uniform $[0, 1]$ or normal $N(0, 1)$ distribution.

**Uniform Output Mapping:**
$$u_i = \frac{\text{rank}(x_i) - 1}{N - 1} \in [0, 1]$$
- **Rank Invariance:** Because it operates purely on ordinal ranks, it completely eliminates the distorting effect of extreme outliers.
- **Finance Perspective:** Widely utilized in quantitative trading for cross-sectional signal ranking (neutralizing outlier earnings shocks).

#### Hand Calculation Drill:
Given trade sizes $x = (10, 20, 30, 100)$ with $N=4$:
- $x = 10 \; (\text{rank } 1) \implies u = \frac{1-1}{3} = 0.00$
- $x = 20 \; (\text{rank } 2) \implies u = \frac{2-1}{3} = 0.333$
- $x = 30 \; (\text{rank } 3) \implies u = \frac{3-1}{3} = 0.667$
- $x = 100 \; (\text{rank } 4) \implies u = \frac{4-1}{3} = 1.00$
- The massive distance $100 - 30 = 70$ is compressed to an equal spacing of $0.333$!


In [ ]:
# 6. QuantileTransformer Verification
from sklearn.preprocessing import QuantileTransformer

trades = np.array([10, 20, 30, 100]).reshape(-1, 1)

qt_uniform = QuantileTransformer(n_quantiles=4, output_distribution='uniform', random_state=42)
qt_normal = QuantileTransformer(n_quantiles=4, output_distribution='normal', random_state=42)

u_transformed = qt_uniform.fit_transform(trades)
n_transformed = qt_normal.fit_transform(trades)

print("=== QuantileTransformer Results ===")
print("Original Trades  :", trades.ravel())
print("Uniform Ranks    :", np.round(u_transformed.ravel(), 3))
print("Gaussian Mapped  :", np.round(n_transformed.ravel(), 3))


### 5.3 Full-Data Practice 3 & 4: Power and Quantile Transforms on Sonar Energy Signals
The Sonar dataset contains 60 continuous acoustic frequency bands. We benchmark how distribution reshaping affects Logistic Regression.


In [ ]:
# Practice 3 & 4: Sonar Energy Transforms
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score

# Self-contained simulation mimicking Sonar energy frequency bands (208 samples, 60 features)
np.random.seed(42)
X_sonar, y_sonar = make_classification(
    n_samples=208, n_features=60, n_informative=15, n_redundant=10,
    weights=[0.53, 0.47], random_state=42
)
# Introduce exponential right-skewness to mimic acoustic energy distributions
X_sonar_skewed = np.exp(X_sonar * 0.7)

X_tr_sn, X_te_sn, y_tr_sn, y_te_sn = train_test_split(
    X_sonar_skewed, y_sonar, test_size=0.3, random_state=42, stratify=y_sonar
)

transforms = {
    "Raw Skewed Energy": None,
    "StandardScaler": StandardScaler(),
    "PowerTransformer (Yeo-Johnson)": PowerTransformer(method='yeo-johnson'),
    "QuantileTransformer (Normal)": QuantileTransformer(n_quantiles=100, output_distribution='normal', random_state=42)
}

print(f"{'Data Transformation Strategy':<32} | {'Test Accuracy':<14}")
print("-" * 50)

for name, tr in transforms.items():
    if tr is None:
        X_tr_t, X_te_t = X_tr_sn, X_te_sn
    else:
        X_tr_t = tr.fit_transform(X_tr_sn)
        X_te_t = tr.transform(X_te_sn)
        
    model = LogisticRegression(max_iter=1000, random_state=42)
    model.fit(X_tr_t, y_tr_sn)
    acc = accuracy_score(y_te_sn, model.predict(X_te_t))
    print(f"{name:<32} | {acc:<14.4f}")


---

## 6. Discretization & Polynomial Feature Expansion

### 6.1 KBinsDiscretizer (Binning Continuous Signals)
Discretization maps continuous features into discrete ordinal bins. This allows linear models to represent non-linear step functions and non-monotonic threshold effects.

**Strategies:**
1. `uniform`: Constant bin widths (Equal-width interval).
2. `quantile`: Equal number of samples per bin (Percentiles).
3. `kmeans`: 1D cluster centers learned via k-means.

#### Hand Calculation Drill:
Given customer incomes $x = (20, 25, 30, 60, 90)$ with $k=3$ bins under `strategy='uniform'`:
- Range $= 90 - 20 = 70$
- Bin Width $= \frac{70}{3} \approx 23.333$
- Bin 0: $[20, 43.333) \implies \text{captures } 20, 25, 30$
- Bin 1: $[43.333, 66.667) \implies \text{captures } 60$
- Bin 2: $[66.667, 90.000] \implies \text{captures } 90$
- Encoded Vector: $(0, 0, 0, 1, 2)^T$


In [ ]:
# 7. KBinsDiscretizer Hands-on Implementation
from sklearn.preprocessing import KBinsDiscretizer

incomes = np.array([20, 25, 30, 60, 90]).reshape(-1, 1)

kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform')
incomes_binned = kbd.fit_transform(incomes)

print("=== KBinsDiscretizer Results ===")
print("Learned Bin Edges :", np.round(kbd.bin_edges_[0], 2))
print("Raw Incomes       :", incomes.ravel())
print("Assigned Bin Index:", incomes_binned.ravel().astype(int))


### 6.2 PolynomialFeatures (Interactions & Non-linear Terms)
Linear models cannot compute multiplicative interactions ($x_1 \times x_2$). `PolynomialFeatures` explicitly manufactures powers and interaction columns.

**Degree 2 Expansion of $(a, b)$:**
$$[a, b] \xrightarrow{\text{degree}=2} [1, a, b, a^2, ab, b^2]$$

#### Hand Calculation Drill:
Given two observations with features $(a, b)$:
- Row 1: $(a=2, b=3) \implies (a=2, b=3, a^2=4, ab=6, b^2=9)$
- Row 2: $(a=4, b=5) \implies (a=4, b=5, a^2=16, ab=20, b^2=25)$


In [ ]:
# 8. PolynomialFeatures Verification
from sklearn.preprocessing import PolynomialFeatures

X_pairs = np.array([[2, 3], [4, 5]])

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_pairs)

print("=== PolynomialFeatures Results ===")
print("Generated Column Names:", poly.get_feature_names_out(['a', 'b']))
print("Expanded Matrix:\n", X_poly)


---

## 7. Transforming the Target Variable ($y$)
In financial asset pricing and real estate valuation, target variables (prices, volume, loss amounts) follow log-normal distributions. Fitting linear models directly on raw $y$ causes squared errors to overfit the massive outliers while underfitting the median majority.

### 7.1 TransformedTargetRegressor Architecture
Scikit-learn's `TransformedTargetRegressor` encapsulates:
1. **Forward Transform:** Transforms $y_{\text{train}}$ via $g(y) = \ln(1 + y)$ prior to model fitting.
2. **Estimator Fit:** Fits the regressor on $(X_{\text{train}}, g(y_{\text{train}}))$.
3. **Inverse Prediction:** When `predict(X_test)` is invoked, it automatically passes raw model predictions through $g^{-1}(\hat{y}) = \exp(\hat{y}) - 1$ back to the original financial unit ($).

#### Hand Calculation Drill:
Given house prices $y = (100, 150, 400)$ (in thousands):
- Transform: $\ln(100) = 4.605, \quad \ln(150) = 5.011, \quad \ln(400) = 5.991$
- Linear model trains on log space.
- Model produces a log forecast $\hat{y}_{\text{log}} = 5.30$.
- Inverted forecast: $\hat{y} = \exp(5.30) \approx 200.34$ thousand dollars.


In [ ]:
# 9. TransformedTargetRegressor Implementation & California Housing Benchmark
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Toy verification
X_toy = np.array([[1], [2], [3], [4]])
y_toy = np.array([100, 150, 400, 1100]) # Exponential growth

ttr_toy = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
)
ttr_toy.fit(X_toy, y_toy)
print("Toy Prediction for X=2.5:", np.round(ttr_toy.predict([[2.5]])[0], 2))

# 2. California Housing Benchmark
X_ca, y_ca = fetch_california_housing(return_X_y=True)
X_tr_ca, X_te_ca, y_tr_ca, y_te_ca = train_test_split(
    X_ca, y_ca, test_size=0.25, random_state=42
)

# Standard Linear Regression without target transform
raw_reg = LinearRegression().fit(X_tr_ca, y_tr_ca)
raw_mae = mean_absolute_error(y_te_ca, raw_reg.predict(X_te_ca))

# TransformedTargetRegressor with log transform
log_reg = TransformedTargetRegressor(
    regressor=LinearRegression(),
    func=np.log1p,
    inverse_func=np.expm1
).fit(X_tr_ca, y_tr_ca)
log_mae = mean_absolute_error(y_te_ca, log_reg.predict(X_te_ca))

print("\n=== California Housing Benchmark Results ===")
print(f"Raw Linear Regression MAE        : {raw_mae:.4f}")
print(f"TransformedTargetRegressor MAE   : {log_mae:.4f} -> Error reduced, heteroskedasticity stabilized!")


---

## 8. Summary & Key Takeaways: Safe Production Architecture
To guarantee zero data leakage in production finance workflows:
1. **Split First:** Always execute `train_test_split` before running any transformation or feature engineering logic.
2. **ColumnTransformer:** Chain feature-specific scalers (e.g., `RobustScaler` for financial ratios, `OneHotEncoder` for exchange symbols).
3. **Pipeline Integration:** Combine preprocessing and downstream estimators into a single callable `Pipeline`.
4. **Target Handling:** Wrap regression estimators with `TransformedTargetRegressor` when targets span orders of magnitude.
